In [0]:
import pyspark.sql.functions as F

dbutils.widgets.text("RUN_DATE", "")
RUN_DATE = dbutils.widgets.get("RUN_DATE").strip()

if not RUN_DATE:
    raise ValueError("RUN_DATE is empty. For manual tests, set RUN_DATE like 2019-12-05.")

print("✅ RUN_DATE =", RUN_DATE)


In [0]:
%run ./06_pipeline_monitoring


In [0]:
silver = spark.table("workspace.default.silver_events").where(F.col("event_date") == F.lit(RUN_DATE))

product_daily = (
    silver
    .where(F.col("product_id").isNotNull())   # keep only valid products
    .groupBy("event_date", "product_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("carts"),
        F.sum(F.when(F.col("event_type") == "remove_from_cart", 1).otherwise(0)).alias("remove_from_cart"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))).alias("revenue"),
        F.countDistinct("user_id").alias("unique_users")
    )
    .withColumn("view_to_purchase_rate", F.when(F.col("views") > 0, F.col("purchases") / F.col("views")).otherwise(F.lit(0.0)))
    .withColumn("cart_to_purchase_rate", F.when(F.col("carts") > 0, F.col("purchases") / F.col("carts")).otherwise(F.lit(0.0)))
)

display(product_daily.limit(20))


In [0]:
target_table = "workspace.default.gold_product_daily_metrics"
try:
    (
      product_daily
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("replaceWhere", f"event_date = DATE('{RUN_DATE}')")
        .saveAsTable(target_table)
    )

    log_run(RUN_DATE, "gold", target_table, "SUCCESS", "Product daily metrics written")
    print("✅ Logged SUCCESS to pipeline_monitoring")

except Exception as e:
    log_run(RUN_DATE, "gold", target_table, "FAILED", str(e)[:1000])
    print("❌ Logged FAILED to pipeline_monitoring")
    raise
